# 09 — Listing-Level Price Prediction (Member 3)

**KPMG slide requirement** ('Price Prediction Modeling' + 'Evaluation and Metrics'). Not consumed by the chatbot. Logic in `src/price_model.py`.

Target `ttm_avg_rate` is modelled as `log1p` (right-skewed) and metrics are inverted to currency units. The data loader globs whatever `*_listings_clean.csv` files are present, so a full local clone (Barcelona + London) retrains the combined model with no code change.

In [1]:
import sys
from pathlib import Path
# Resolve repo root whether run from notebooks/ or repo root
_here = Path.cwd()
ROOT = _here if (_here / 'src').exists() else _here.parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, joblib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_io import PROCESSED_DIR
print('repo root:', ROOT)

repo root: /home/claude/KPMG_Airbnb_Capstone


In [2]:
from src import price_model as pm
data = pm.load_price_frame()
print('rows:', len(data), '| cities:', sorted(data.city.unique()))

rows: 1825 | cities: ['barcelona']


## Train linear / random forest / gradient boosting; select best by hold-out R²

In [3]:
res = pm.train_and_select(data)
print('best:', res['best_name'], '| train', res['n_train'], 'test', res['n_test'], '| cities', res['cities'])
res['metrics']

best: gradient_boosting | train 1460 test 365 | cities ['barcelona']


,rmse,mae,r2,cv_r2_mean,cv_r2_std
linear,118.48,72.77,0.5785,0.5381,0.0351
random_forest,111.31,68.65,0.6281,0.5492,0.0230
gradient_boosting,102.05,65.94,0.6873,0.5675,0.0236


## Feature importance (best tree model)

In [4]:
res['feature_importance'].head(15)

,feature,importance
0,num__beds,0.256310
1,cat__listing_type_Private room in rental unit,0.189863
2,num__guests,0.187852
3,num__baths,0.056299
4,num__host_listing_count,0.044499
5,num__bedrooms,0.031458
6,cat__room_type_entire_home,0.027798
7,cat__neighborhood_Eixample,0.021695
8,cat__listing_type_Room in hotel,0.020032
9,cat__listing_type_Room in boutique hotel,0.016986


## Save the model

In [5]:
(ROOT/'models').mkdir(exist_ok=True)
joblib.dump(res['best_pipeline'], ROOT/'models'/'price_model.joblib')
print('saved models/price_model.joblib')
# sanity: reload and predict one row
m = joblib.load(ROOT/'models'/'price_model.joblib')
X,_,_ = pm.build_xy(data)
print('sample prediction (currency):', round(float(np.expm1(m.predict(X.head(1))[0])),2))

saved models/price_model.joblib
sample prediction (currency): 68.78
